# Valuation of Performance Share Units (PSUs) under IFRS 2: A Technical Perspective

## 1. Introduction and The Vesting Framework
In equity compensation (under IFRS 2 or ASC 718), "vesting" represents the process by which a beneficiary earns the non-forfeitable right to receive shares. This right is contingent upon satisfying specific requirements, which fall into three main categories:

* **Service Conditions:** Require only that the employee continues to provide services to the entity for a specified period (retention).
* **Market Conditions:** Linked to the performance of the company's equity (or related derivatives), such as achieving a Target Price or a specific Relative Total Shareholder Return (TSR) against a peer group.
* **Non-Market Performance Conditions:** Tied to internal financial, operational, or strategic metrics (e.g., EPS, EBITDA targets, or specific project milestones).

### 1.1 The Valuation Asymmetry
IFRS 2 dictates a strict asymmetry in handling these conditions. Market conditions are incorporated directly into the grant-date fair value using quantitative models under the risk-neutral measure ($Q$). Conversely, Non-Market and Service conditions do not affect the initial fair value unit but act stochastically on the estimated number of instruments expected to vest under the real-world measure ($P$), allowing for subsequent true-up adjustments to the income statement.

### 1.2 Payout Multipliers and Path-Dependency
In advanced PSU structures, performance conditions rarely act as simple binary triggers (cliff vesting). Instead, the contract often introduces a **multiplicative component** defined by a payout curve. The payoff is scaled based on the degree of achievement: e.g., 50% vesting at a minimum threshold, 100% at target, and up to a 200% cap for outperformance.

Crucially, these conditions make the structure deeply **path-dependent**. Vesting and multipliers often depend on the entire price evolution over time rather than just the terminal value ($S_T$). Examples include averaging mechanisms (Asian-style features over the last 60-90 days), high-water marks, or laddered barriers. This path dependency renders closed-form solutions (like the Black-Scholes formula) inapplicable, making Monte Carlo simulation the standard numerical approach.

## 2. Technical Framework for PSU Valuation
PSUs are contracts whose payoff depends on one or more stochastic drivers and an indicator of vesting. The grant-date fair value is:
$$FV_{grant} = E_Q \left[ D(0,T) \cdot H(S_{\cdot}, M_{\cdot}) \cdot 1_{\{Vesting\}} \right]$$
where
- $D(0,T) = \exp\left(-\int_0^T r_t \, dt\right)$
- $S_{\cdot}$ denotes tradable market factors
- $M_{\cdot}$ denotes non-market metrics
- $1_{\{Vesting\}} = 1$ if all vesting conditions are satisfied

For equity settlement, payoff can be written as
$$H = S_T \cdot g(S_{\cdot}, M_{\cdot})$$
or, when the award is expressed as a share count,
$$H = N(S_{\cdot}, M_{\cdot}) \cdot S_T$$
with payout functions $g(\cdot)$ and $N(\cdot)$ incorporating caps, floors, linear/curved payout schedules, and partial vesting.

### 2.1 Market versus Non-Market Conditions
Market conditions:
- valued under risk-neutral measure $Q$
- modeled using SDEs such as
$$dS_t = S_t (r - q) \, dt + S_t \sigma \, dW_t^Q$$

Non-market conditions:
- reflected through probability of attainment $p_{\text{attain}}$
- not remeasured as market instruments
- incorporated as an adjustment factor:
$$FV_{grant} = E_Q \left[ D(0,T) \cdot H(S_{\cdot}) \cdot 1_{\{\text{MarketConditions}\}} \right] \cdot p_{\text{nonmarket}}$$

When non-market metrics are stochastic:
$$FV_{grant} = E \left[ D(0,T) \cdot H(S_{\cdot}, M_{\cdot}) \cdot 1_{\{Vesting\}} \right]$$
with $M_{\cdot}$ simulated under a real-world measure $P$ or calibrated to empirical distributions.

### 2.2 Vesting and Service Probabilities
Service conditions are modeled via a survival function:
$$P_{\text{service}}(t) = \exp\left(-\int_0^t \lambda(u) \, du\right)$$

For graded vesting over tranches $\{t_i\}$,
$$FV_{grant} = \sum_{i} E_Q \left[ D(0,t_i) \cdot H_i \cdot 1_{\{Vesting_i\}} \right] \cdot P_{\text{service}}(t_i)$$

For discrete vesting with hurdle thresholds $H_k$ and payout curves $f(x)$,
$$\text{Payoff}_i = f(\text{Performance}_i) \cdot S_{t_i}$$

## 3. Monte Carlo Justification
Monte Carlo is necessary because PSUs often feature:
- path-dependent vesting (averaging, high-water marks, multi-period TSR)
- multidimensional underlyings (peer group returns, correlated KPIs)
- discontinuous payoffs (laddered or barrier-based vesting)
- mixed conditions requiring joint simulation of $S$ and $M$

The numerical estimator is:
$$FV_N = \frac{1}{N} \sum_{j=1}^N D_j \cdot H_j \cdot 1_{\{Vesting\}_j}$$
with standard error $\approx \sigma_H / \sqrt{N}$ and potential variance reduction using antithetic sampling, control variates, or quasi-Monte Carlo.

## 4. Formal Modelling of Conditions
Service condition:
- hazard $\lambda(t)$, survival $S_{\text{srv}}(t) = \exp\left(-\int_0^t \lambda(u) \, du\right)$

Market condition:
- indicator $1_{\{\text{Market}\}} = 1$ if relative TSR or index-based hurdle is met
- example for relative TSR rank:
$$1_{\{Rank \ge \alpha\}} = 1_{\{\mathrm{TSR}_{company} / \mathrm{TSR}_{peer} \ge \text{threshold}\}}$$

Non-market condition:
- attainment probability $p_{\text{attain}} = P_P(M_T \ge \text{target})$
- for multiple conditions:
$$1_{\{Vesting\}} = 1_{\{\text{Market}\}} \cdot 1_{\{\text{NonMarket}\}} \cdot 1_{\{\text{Service}\}}$$
and if non-market metrics are stochastic,
$$E[1_{\{\text{NonMarket}\}}] = \int 1_{\{m \text{ meets target}\}} \, f_M(m) \, dm$$

## 5. Accounting Implications
Grant-date measurement:
- $FV_{grant}$ is recognized over the service period
- expense for period $t$:
$$\text{Expense}_t = FV_{grant} \times \Delta P_{\text{service}}(t)$$

For a total grant of $N$ awards:
$$\text{Total\_cost} = N \times FV_{grant}$$

If vesting is graded, allocate expense by tranche weights and expected survival.

Changes in estimates:
- service forfeiture updates affect future expense
- market conditions remain priced in at grant date
- non-market conditions are reflected only through probability updates when required by IFRS 2 guidance

## 6. Implementation Principles
- simulate $S_t$ under risk-neutral dynamics calibrated to implied vol and dividend yield
- simulate $M_t$ under real-world models or scenario distributions
- apply correlation structure between market and non-market drivers if dependencies exist
- use variance reduction and ensure enough paths for target standard error
- verify with sensitivity analysis on $\sigma$, $r$, $q$, correlation, $p_{\text{attain}}$, and vesting hazards

In [ ]:
import numpy as np

def simulate_stock_paths(S0, r, q, sigma, T, n_steps, n_paths, seed=42):
    np.random.seed(seed)
    dt = T / n_steps
    increments = np.random.normal(size=(n_paths, n_steps)) * np.sqrt(dt)
    S = np.empty((n_paths, n_steps + 1))
    S[:, 0] = S0
    for t in range(n_steps):
        S[:, t + 1] = S[:, t] * np.exp((r - q - 0.5 * sigma**2) * dt + sigma * increments[:, t])
    return S

# Parametri di esempio
S0 = 100.0          # prezzo azione iniziale
r = 0.03            # tasso privo di rischio
q = 0.02            # dividend yield
sigma = 0.25        # volatilità
T = 3.0             # maturità in anni
n_steps = 252 * 3   # passi giornalieri su 3 anni
n_paths = 100000    # numero percorsi Monte Carlo

# Condizioni
hurdle_price = 105.0      # soglia per la market condition
p_nonmarket = 0.80        # probabilità di soddisfare la non-market condition

paths = simulate_stock_paths(S0, r, q, sigma, T, n_steps, n_paths)
avg_last_60 = paths[:, -60:].mean(axis=1)
terminal_price = paths[:, -1]

market_condition = avg_last_60 >= hurdle_price
payoff = np.where(market_condition, terminal_price, 0.0)

discount = np.exp(-r * T)
fv_grant = discount * payoff.mean() * p_nonmarket

print("FV grant (con market + non-market):", fv_grant)
print("Probabilità market condition:", market_condition.mean())